In [1]:
import pandas as pd
import numpy as np

# Завантаження датасету
df = pd.read_csv(
    'household_power_consumption.txt',
    sep=';',
    na_values=['?'],
    low_memory=False
)

# Об'єднуємо Date і Time в один стовпець datetime
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df = df.drop(columns=['Date', 'Time'])

# Заповнюємо пропуски середнім значенням
df = df.fillna(df.mean(numeric_only=True))

print(f"Розмір датасету: {df.shape}")
print(f"Пропуски: {df.isnull().sum().sum()}")
df.head()

Розмір датасету: (2075259, 8)
Пропуски: 0


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
0,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


In [2]:
import time

# 1. Всі записи де загальна активна потужність > 5 кВт
def query1(df):
    return df[df['Global_active_power'] > 5]

start = time.time()
result1 = query1(df)
print(f"Час: {time.time() - start:.4f} сек")
print(f"Записів де потужність > 5 кВт: {len(result1)}")
result1.head()

Час: 0.0050 сек
Записів де потужність > 5 кВт: 17547


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
11,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00
12,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00


In [3]:
import time

# 2. Сила струму 19-20А, пральна+холодильник > бойлер+кондиціонер
def query2(df):
    subset = df[(df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)]
    return subset[subset['Sub_metering_2'] > subset['Sub_metering_3']]

start = time.time()
result2 = query2(df)
print(f"Час: {time.time() - start:.4f} сек")
print(f"Знайдено записів: {len(result2)}")
result2.head()

Час: 0.0090 сек
Знайдено записів: 2509


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
45,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00
475,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00
476,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00


In [4]:
# 3. Випадкові 500000 записів, середні по групах
def query3(df):
    sample = df.sample(n=500000, replace=False)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

start = time.time()
result3 = query3(df)
print(f"Час: {time.time() - start:.4f} сек")
print("Середні значення груп споживання (500k записів):")
print(result3)

Час: 0.1153 сек
Середні значення груп споживання (500k записів):
Sub_metering_1    1.132789
Sub_metering_2    1.303174
Sub_metering_3    6.467467
dtype: float64


In [5]:
# 4. Після 18:00, > 6 кВт, група 2 найбільша, кожен 3-й з першої і 4-й з другої половини
def query4(df):
    subset = df[(df['datetime'].dt.hour >= 18) & (df['Global_active_power'] > 6)]
    subset = subset[(subset['Sub_metering_2'] > subset['Sub_metering_1']) & 
                    (subset['Sub_metering_2'] > subset['Sub_metering_3'])]
    mid = len(subset) // 2
    first_half = subset.iloc[:mid].iloc[::3]
    second_half = subset.iloc[mid:].iloc[::4]
    return pd.concat([first_half, second_half])

start = time.time()
result4 = query4(df)
print(f"Час: {time.time() - start:.4f} сек")
print(f"Знайдено записів: {len(result4)}")
result4.head()

Час: 0.0412 сек
Знайдено записів: 310


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
41,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00
44,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00
17494,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00
17498,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00
17501,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00


In [6]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

# Нормалізація (Min-Max)
scaler_minmax = MinMaxScaler()
df_normalized = df.copy()
df_normalized[numeric_cols] = scaler_minmax.fit_transform(df[numeric_cols])

print("Нормалізований датасет:")
df_normalized[numeric_cols].head()

Нормалізований датасет:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


In [7]:
import sys
!{sys.executable} -m pip install scikit-learn


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

# Нормалізація (Min-Max)
scaler_minmax = MinMaxScaler()
df_normalized = df.copy()
df_normalized[numeric_cols] = scaler_minmax.fit_transform(df[numeric_cols])

print("Нормалізований датасет:")
df_normalized[numeric_cols].head()

Нормалізований датасет:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


In [10]:
# Стандартизація (Z-score)
scaler_std = StandardScaler()
df_standardized = df.copy()
df_standardized[numeric_cols] = scaler_std.fit_transform(df[numeric_cols])

print("Стандартизований датасет:")
df_standardized[numeric_cols].head()

Стандартизований датасет:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2.973749,2.627217,-1.863517,3.118369,-0.183489,-0.051598,1.257315
1,4.062593,2.787911,-2.239335,4.159920,-0.183489,-0.051598,1.138043
2,4.075918,3.341412,-2.344937,4.159920,-0.183489,0.121249,1.257315
3,4.089243,3.377122,-2.205170,4.159920,-0.183489,-0.051598,1.257315
4,2.450266,3.609235,-1.602618,2.529666,-0.183489,-0.051598,1.257315


In [11]:
# Коефіцієнт Пірсона та Спірмена
from scipy import stats

col1 = 'Global_active_power'
col2 = 'Global_intensity'

pearson_r, pearson_p = stats.pearsonr(df[col1], df[col2])
spearman_r, spearman_p = stats.spearmanr(df[col1], df[col2])

print(f"Пірсон:  r = {pearson_r:.4f}, p-value = {pearson_p:.4e}")
print(f"Спірмен: r = {spearman_r:.4f}, p-value = {spearman_p:.4e}")

Пірсон:  r = 0.9989, p-value = 0.0000e+00
Спірмен: r = 0.9955, p-value = 0.0000e+00


In [12]:
# One Hot Encoding категоріального атрибута (година доби)
df['hour_category'] = pd.cut(df['datetime'].dt.hour, 
                              bins=[0, 6, 12, 18, 24], 
                              labels=['ніч', 'ранок', 'день', 'вечір'],
                              right=False)

df_encoded = pd.get_dummies(df, columns=['hour_category'])
print("Після One Hot Encoding:")
df_encoded.filter(like='hour_category').head(10)

Після One Hot Encoding:


,hour_category_ніч,hour_category_ранок,hour_category_день,hour_category_вечір
0,False,False,True,False
1,False,False,True,False
2,False,False,True,False
3,False,False,True,False
4,False,False,True,False
5,False,False,True,False
6,False,False,True,False
7,False,False,True,False
8,False,False,True,False
9,False,False,True,False
